# Bloodflow: does aggregation turn Poiseuille flow into plug flow?

A toy model: rigid disks in a 2D channel under Poiseuille flow. Turn on a
short-range (Morse) attraction and the disks aggregate into clusters that
move as a plug, carrying slow near-wall particles along — cutting the long
tail of transit times.

This notebook runs entirely in your browser (JupyterLite + Pyodide).
The two runs below take about a minute.

In [ ]:
import numpy as np
from dataclasses import replace
from bloodflow import Params, cluster_stats, simulate, transit_summary

base = Params(
    N=60, a=1.0, H=16.0, L=40.0, eta=1.0, Umax=1.0,
    kappa=2.0, eps_rep=1.0, kT=0.0, dt=0.02, seed=1,
)
results = {}
for eps in (0.0, 2.0):
    p = replace(base, eps=eps)
    res = simulate(p, n_steps=8000, equil_steps=500)
    s = transit_summary(res.transit, res.wall_time)
    c = cluster_stats(res.final_pos, cutoff=2.0 * p.a + 0.5 / p.kappa)
    results[eps] = (res, s, c)
    print(f"eps={eps}: mean={s['mean']:.1f}  median={s['median']:.1f}  "
          f"p95={s['p95']:.1f}  wall residence={s['mean_wall_residence']:.1f}  "
          f"mean cluster={c['mean_cluster_size']:.1f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
for eps, color in ((0.0, "tab:blue"), (2.0, "tab:orange")):
    res, s, c = results[eps]
    done = res.transit[~np.isnan(res.transit)]
    ax.hist(done, bins=25, alpha=0.6, color=color,
            label=f"eps={eps} (p95={s['p95']:.0f})")
ax.set_xlabel("transit time")
ax.set_ylabel("particles")
ax.legend()
ax.set_title("Transit-time distribution: no attraction vs aggregation")
plt.show()

## Takeaway

With attraction on (eps = 2), the long tail of slow transits shrinks
(p95 138 → 75), near-wall residence collapses (35 → 4), and clusters grow
(mean size 1.5 → 15). The aggregates move as a plug.

See also: the [pulsatile-flow video](videos/bloodflow-pulsatile.mp4)
(heartbeat version — the plug breathes but survives) and the
[full write-up](https://github.com/marcocaggioni/bloodflow/blob/main/README.md).

## Export an MP4 in the browser

JupyterLite ships no ffmpeg binary, so a notebook cannot normally write a
video file. This section works around that with
[ffmpeg.wasm](https://ffmpegwasm.netlify.app/) — FFmpeg compiled to
WebAssembly, driven from Python through Pyodide's JavaScript bridge.

The ffmpeg files are vendored in this repo under `vendor/ffmpeg/` and
served from the same site as the notebook (no CDN): the single-threaded
core is used on purpose, since GitHub Pages sends no cross-origin-isolation
headers, so the multi-threaded core cannot load here. The first run
downloads ~31 MB of WASM; the browser caches it afterwards. (The earlier
CDN version failed live: the browser blocks the cross-origin worker file,
so same-origin loading is required.)

Run the two cells below: the first renders the eps = 2 run as 90 PNG
frames (3 s at 30 fps), the second encodes them with libx264 and saves
`bloodflow-export.mp4` next to this notebook — it appears in the file
browser (right-click → Download) and plays inline below. The temporary
PNG frames are deleted from ffmpeg's in-memory filesystem after encoding;
the simulation data stays in the notebook.

In [ ]:
# Render the eps = 2 run as PNG frames (kept in memory) for the MP4 export below.
# Same seed / equilibration as the sweep above, so this is that run's plug forming,
# re-run short (500 equil + 1800 steps) just for the animation.
import io

import numpy as np
import matplotlib.pyplot as plt
from bloodflow import Params, core

FPS = 30
N_FRAMES = 90
STRIDE = 20            # steps between frames (0.4 time units per frame)
EQUIL_STEPS = 500

p_anim = Params(N=60, a=1.0, H=16.0, L=40.0, eta=1.0, Umax=1.0,
                kappa=2.0, eps_rep=1.0, kT=0.0, dt=0.02, seed=1, eps=2.0)
rng = np.random.default_rng(p_anim.seed)
pos = core.init_positions(p_anim, rng)
for _ in range(EQUIL_STEPS):
    core._step(pos, p_anim, rng, 0.0)

fig, ax = plt.subplots(figsize=(10, 4), dpi=80)
ax.set_xlim(0, p_anim.L)
ax.set_ylim(0, p_anim.H)
ax.set_aspect("equal")
ax.set_xlabel("x (flow direction)")
ax.set_ylabel("y (channel width)")
ax.set_title("eps = 2 (aggregation) — plug flow")
fig.canvas.draw()
# Marker area (points^2) matching a disk of radius a in data units.
p0 = ax.transData.transform((0.0, 0.0))
p1 = ax.transData.transform((2.0 * p_anim.a, 0.0))
r_px = abs(p1[0] - p0[0]) / 2.0
size = (2.0 * r_px * 72.0 / fig.dpi) ** 2
scat = ax.scatter(pos[:, 0], pos[:, 1], s=size, c="tab:orange", edgecolors="none")
time_text = ax.text(0.01, 0.95, "", transform=ax.transAxes, va="top", fontsize=10)

frames_png = []
for frame in range(N_FRAMES):
    if frame > 0:
        for _ in range(STRIDE):
            core._step(pos, p_anim, rng, 0.0)
    scat.set_offsets(pos)
    time_text.set_text(f"t = {(EQUIL_STEPS + frame * STRIDE) * p_anim.dt:.1f}")
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    frames_png.append(buf.getvalue())
plt.close(fig)
print(f"rendered {len(frames_png)} frames "
      f"({len(frames_png) / FPS:.1f} s at {FPS} fps, eps = 2.0)")

In [ ]:
# Encode the frames above into an MP4, entirely in the browser.
# JupyterLite has no ffmpeg binary, so this cell loads ffmpeg.wasm
# (FFmpeg compiled to WebAssembly) and drives it from Python through
# Pyodide's JavaScript bridge. The ffmpeg files are vendored in this
# repo under vendor/ffmpeg/ and served from the same site as the
# notebook — no CDN. (The CDN version failed live: the browser blocks
# the cross-origin worker file, so same-origin loading is required.)
# Single-threaded core on purpose: GitHub Pages sends no
# cross-origin-isolation headers, so the multi-threaded core cannot
# load here. First run downloads ~31 MB; the browser caches it
# afterwards.
import re

from pyodide.code import run_js
from pyodide.ffi import to_js

try:
    n_frames = len(frames_png)
    fps = FPS
except NameError:
    raise RuntimeError(
        "Run the 'Render frames' cell above first — it defines frames_png and FPS."
    )

# ffmpeg.wasm 0.12.15 (@ffmpeg/ffmpeg) + single-threaded core 0.12.10,
# vendored in vendor/ffmpeg/. The site root is derived from the kernel
# worker's own URL (see below): the worker script lives under
# <site>/extensions/, so everything before that is the site root.
FFMPEG_ESM = "ffmpeg-esm/index.js"   # @ffmpeg/ffmpeg ESM entry
FFMPEG_WORKER = "ffmpeg-esm/worker.js"
CORE_JS = "core/ffmpeg-core.js"      # @ffmpeg/core, single-threaded UMD build
CORE_WASM = "core/ffmpeg-core.wasm"
OUT_NAME = "bloodflow-export.mp4"

# The Pyodide kernel runs in a Web Worker, where `window` does not exist,
# so the site root is derived from the worker script's own URL: the worker
# lives under <site>/extensions/@jupyterlite/pyodide-kernel-extension/static/,
# so everything before /extensions/ is the site root (/bloodflow/ on GitHub
# Pages, / on a local JupyterLite at the domain root).
worker_pathname = run_js("self.location.pathname")
m = re.match(r"^(.*?)/extensions/", worker_pathname)
if m is not None:
    site_base = m.group(1) + "/"
else:
    # Fallback (e.g. a main-thread kernel): the lab/tree page URL.
    m = re.match(r"^(.*)/(lab|tree)(?:/|$)", worker_pathname)
    site_base = (m.group(1) if m else "") + "/"
vendor_base = f"{site_base}files/vendor/ffmpeg"

# The import and load() URLs below must be fully qualified, not
# site-relative. The kernel worker itself is served from this site, so
# self.location gives the right origin — but dynamic import() inside
# run_js() resolves relative specifiers against the Pyodide runtime
# module, which loads from cdn.jsdelivr.net (the extension's default
# PYODIDE_CDN_URL), so a path like /bloodflow/files/... was fetched
# from the CDN and 404'd. Resolving the path against the worker's own
# location first yields an absolute site URL, which import()/fetch()
# then use as-is. GitHub Pages serves Access-Control-Allow-Origin: *
# on these files, so the cross-origin fetch from the CDN-origin worker
# realm is allowed. This keeps working on a local JupyterLite too
# (the origin just becomes localhost).
vendor_base_url = run_js(
    f"new URL('{vendor_base}', self.location.href).href"
)
print(f"loading ffmpeg.wasm from {vendor_base_url} ...")

ffmpeg_mod = await run_js(f"import('{vendor_base_url}/{FFMPEG_ESM}')")
ffmpeg = ffmpeg_mod.FFmpeg.new()
print("loading ffmpeg.wasm core (vendored in the repo; first run downloads ~31 MB)...")
await ffmpeg.load({
    "classWorkerURL": f"{vendor_base_url}/{FFMPEG_WORKER}",
    "coreURL": f"{vendor_base_url}/{CORE_JS}",
    "wasmURL": f"{vendor_base_url}/{CORE_WASM}",
})
print("ffmpeg.wasm loaded")

for i, png in enumerate(frames_png):
    # to_js: bytes must arrive as a real JS Uint8Array - a PyProxy
    # cannot be posted to ffmpeg's worker (DataCloneError).
    await ffmpeg.writeFile(f"frame_{i:04d}.png", to_js(png))
frames_png.clear()  # free the Python-side copies; the frames now live in ffmpeg's memory FS
print(f"wrote {n_frames} frames to ffmpeg's in-memory filesystem")

# to_js again: the arg list must be a real JS Array - a PyProxy list
# cannot be posted to ffmpeg's worker either (same DataCloneError).
rc = await ffmpeg.exec(to_js([
    "-y", "-framerate", str(fps), "-i", "frame_%04d.png",
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20",
    "-movflags", "+faststart", OUT_NAME,
]))
if rc != 0:
    raise RuntimeError(f"ffmpeg exited with code {rc}")
mp4_bytes = bytes((await ffmpeg.readFile(OUT_NAME)).to_py())
with open(OUT_NAME, "wb") as f:
    f.write(mp4_bytes)
print(f"wrote {OUT_NAME} ({len(mp4_bytes)} bytes) — "
      "find it in the file browser (right-click → Download)")

# Cleanup: delete the temporary frames (and the encoded copy) from
# ffmpeg's in-memory filesystem. The simulation data (results) and the
# saved MP4 are untouched.
for i in range(n_frames):
    await ffmpeg.deleteFile(f"frame_{i:04d}.png")
await ffmpeg.deleteFile(OUT_NAME)
print(f"deleted {n_frames} temporary frames from ffmpeg's filesystem")

from IPython.display import Video, display
try:
    display(Video(OUT_NAME, embed=True))
except Exception as exc:  # inline preview is a bonus; the file is already saved
    print(f"(inline preview unavailable: {exc} — download {OUT_NAME} from the file browser)")